In [ ]:
!pip install zstandard # dataset compression

In [ ]:
import pandas as pd
import numpy as np

## Download ANES Data and Unpack

Download the ANES 2024 dataset from here: https://electionstudies.org/data-center/2024-time-series-study/, then unpack it and put the `anes_timeseries_2024_csv_20260519.csv` file into the data folder.

(I tried to automate this, but the ANES website was blocking `wget`)

## ANES Data Exploration

In [ ]:
df = pd.read_csv('data/anes_timeseries_2024_csv_20260519.csv', low_memory=False)
df

Variables to note from the codebook below. All persona variables and the outcome come from the **pre-election** interview (see `docs/anes_variable_consistency_pre.md`). I tried to match them as closely as possible to the 2016 / 2020 versions of the ANES that Argyle et al. (2023) used.

**Note that the patriotism (American flag) question was only included in the 2016 version of the ANES, but is missing in 2020 and 2024.**

- `V240001` Case ID
- `V240002a` Interview mode (3 = paper questionnaire, PAPI)
- `V240107a` Pre-election full sample weight
---
- `V241458x` PRE: SUMMARY: RESPONDENT AGE ON ELECTION DAY
- `V241004` PRE: HOW OFTEN DOES R PAY ATTENTION TO POLITICS AND ELECTIONS
- `V241501x` PRE: SUMMARY: R SELF-IDENTIFIED RACE/ETHNICITY
- `V241177` PRE: 7PT SCALE LIBERAL-CONSERVATIVE SELF-PLACEMENT
- `V241227x` PRE: SUMMARY: PARTY ID
- `V241439` PRE: EVER ATTEND CHURCH OR RELIGIOUS SERVICES
- `V241550` PRE: WHAT IS R'S SEX? [REVISED] (as in 2020; also asked on the paper questionnaire)
- `V241005` PRE: HOW INTERESTED IN FOLLOWING CAMPAIGNS
- `V243001` SAMPLE: SAMPLE LOCATION STATE POSTAL ABBREVIATION
---
- `V241075x` PRE: SUMMARY: PARTY OF PRE-ELECTION PRESIDENTIAL VOTE/INTENT/PREFERENCE
- `V241102x` PRE: SUMMARY: HOW LIKELY R IS TO VOTE
---
Only kept for a secondary validation against the reported vote (never used in prompts):
- `V240107b` Post-election full sample weight
- `V242095x` PRE-POST: SUMMARY: VOTER TURNOUT IN 2024
- `V242096x` PRE-POST: SUMMARY: 2024 PRESIDENTIAL VOTE

In [ ]:
cols = {
  'V240001': 'anes_id',
  'V240002a': 'mode',
  'V240107a': 'pre_weight',
  'V240107b': 'post_weight',
  'V241458x': 'age',
  'V241004': 'political_attention',
  'V241501x': 'race',
  'V241177': 'ideology',
  'V241227x': 'party',
  'V241439': 'attend_church',
  'V241550': 'gender',
  'V241005': 'campaign_interest',
  'V243001': 'state',
  'V241075x': 'vote_intent_summary',
  'V241102x': 'turnout_likelihood',
  'V242095x': 'reported_turnout',
  'V242096x': 'reported_vote',
}

short_df = df[cols.keys()]
short_df = short_df.rename(columns=cols)
short_df

In [ ]:
cols.values()

## ANES Preprocessing

In [ ]:
preproc_df = short_df.copy()

preproc_df['pre_weight'] = pd.to_numeric(preproc_df['pre_weight'], errors='coerce') # convert missing weights to np.nan
preproc_df['post_weight'] = pd.to_numeric(preproc_df['post_weight'], errors='coerce') # convert missing weights to np.nan

preproc_df['age'] = preproc_df.age.replace({-2: np.nan})
# Template wording: "I ..."
preproc_df['political_attention'] = preproc_df.political_attention.case_when([
    (preproc_df.political_attention == 1, "always pay attention to what's going on in government and politics"),
    (preproc_df.political_attention == 2, "pay attention to what's going on in government and politics most of the time"),
    (preproc_df.political_attention == 3, "pay attention to what's going on in government and politics about half the time"),
    (preproc_df.political_attention == 4, "pay attention to what's going on in government and politics some of the time"),
    (preproc_df.political_attention == 5, "never pay attention to what's going on in government and politics"),
    ([True]*len(preproc_df), np.nan), # catch-all, missing values
])
preproc_df['race'] = preproc_df.race.case_when([
    (preproc_df.race == 1, 'White'),
    (preproc_df.race == 2, 'Black'),
    (preproc_df.race == 3, 'Hispanic'),
    (preproc_df.race == 4, 'Asian'),
    (preproc_df.race == 5, 'Native American'),
    (preproc_df.race == 6, 'of multiple races'), # Template wording: "Racially, I am..."
    ([True]*len(preproc_df), np.nan), # catch-all, missing values
])
preproc_df['ideology'] = preproc_df.ideology.case_when([
    (preproc_df.ideology == 1, 'extremely liberal'),
    (preproc_df.ideology == 2, 'liberal'),
    (preproc_df.ideology == 3, 'slightly liberal'),
    (preproc_df.ideology == 4, 'moderate'),
    (preproc_df.ideology == 5, 'slightly conservative'),
    (preproc_df.ideology == 6, 'conservative'),
    (preproc_df.ideology == 7, 'extremely conservative'),
    (preproc_df.ideology == 99, np.nan), # "Haven’t thought much about this"
    ([True]*len(preproc_df), np.nan), # catch-all, missing values
])
preproc_df['party'] = preproc_df.party.case_when([
    (preproc_df.party == 1, 'a strong Democrat'),
    (preproc_df.party == 2, 'a weak Democrat'),
    (preproc_df.party == 3, 'an independent who leans Democratic'),
    (preproc_df.party == 4, 'an independent'),
    (preproc_df.party == 5, 'an independent who leans Republican'),
    (preproc_df.party == 6, 'a weak Republican'),
    (preproc_df.party == 7, 'a strong Republican'),
    ([True]*len(preproc_df), np.nan), # catch-all, missing values
])
preproc_df['attend_church'] = preproc_df.attend_church.case_when([
    (preproc_df.attend_church == 1, True),
    (preproc_df.attend_church == 2, False),
    ([True]*len(preproc_df), np.nan), # catch-all, missing values
])
preproc_df['gender'] = preproc_df.gender.case_when([
    (preproc_df.gender == 1, 'male'),
    (preproc_df.gender == 2, 'female'),
    ([True]*len(preproc_df), np.nan), # catch-all, missing values
])
# Template wording: "I have been ... interested in the political campaigns so far this year."
preproc_df['campaign_interest'] = preproc_df.campaign_interest.case_when([
    (preproc_df.campaign_interest == 1, 'very much'),
    (preproc_df.campaign_interest == 2, 'somewhat'),
    (preproc_df.campaign_interest == 3, 'not much'),
    ([True]*len(preproc_df), np.nan), # catch-all, missing values, including PAPI (not asked)
])
preproc_df['state'] = preproc_df.state.map({
    'AL': 'Alabama',
    'AK': 'Alaska',
    'AZ': 'Arizona',
    'AR': 'Arkansas',
    'CA': 'California',
    'CO': 'Colorado',
    'CT': 'Connecticut',
    'DE': 'Delaware',
    'DC': 'Washington DC',
    'FL': 'Florida',
    'GA': 'Georgia',
    'HI': 'Hawaii',
    'ID': 'Idaho',
    'IL': 'Illinois',
    'IN': 'Indiana',
    'IA': 'Iowa',
    'KS': 'Kansas',
    'KY': 'Kentucky',
    'LA': 'Louisiana',
    'ME': 'Maine',
    'MD': 'Maryland',
    'MA': 'Massachusetts',
    'MI': 'Michigan',
    'MN': 'Minnesota',
    'MS': 'Mississippi',
    'MO': 'Missouri',
    'MT': 'Montana',
    'NE': 'Nebraska',
    'NV': 'Nevada',
    'NH': 'New Hampshire',
    'NJ': 'New Jersey',
    'NM': 'New Mexico',
    'NY': 'New York',
    'NC': 'North Carolina',
    'ND': 'North Dakota',
    'OH': 'Ohio',
    'OK': 'Oklahoma',
    'OR': 'Oregon',
    'PA': 'Pennsylvania',
    'RI': 'Rhode Island',
    'SC': 'South Carolina',
    'SD': 'South Dakota',
    'TN': 'Tennessee',
    'TX': 'Texas',
    'UT': 'Utah',
    'VT': 'Vermont',
    'VA': 'Virginia',
    'WA': 'Washington',
    'WV': 'West Virginia',
    'WI': 'Wisconsin',
    'WY': 'Wyoming',
}) # missing values are mapped to np.nan

# Pre-election vote intention (Option A in docs/anes_variable_consistency_pre.md)
already_voted = preproc_df.vote_intent_summary.isin([10, 11, 12])
papi = preproc_df['mode'] == 3 # paper questionnaire has no intent question, only a preference
low_turnout = preproc_df.turnout_likelihood.isin([5, 6]) # slightly likely / not likely at all
preproc_df['vote_choice'] = preproc_df.vote_intent_summary.case_when([
    (~already_voted & low_turnout, 'Non-voter'),
    (preproc_df.vote_intent_summary.isin([10, 20]) | (papi & (preproc_df.vote_intent_summary == 30)), 'Harris'),
    (preproc_df.vote_intent_summary.isin([11, 21]) | (papi & (preproc_df.vote_intent_summary == 31)), 'Trump'),
    ([True]*len(preproc_df), np.nan), # catch-all: other candidates, preference only, missing values
])

# Reported vote (pre-post summary), only for a secondary validation
preproc_df['reported_vote'] = preproc_df.reported_vote.case_when([
    (preproc_df.reported_vote == 1, 'Harris'),
    (preproc_df.reported_vote == 2, 'Trump'),
    (preproc_df.reported_turnout == 0, 'Non-voter'),
    ([True]*len(preproc_df), np.nan), # catch-all, missing values
])
preproc_df

In [ ]:
vote_intention_df = preproc_df.dropna(subset='vote_choice')
vote_intention_df.info()

In [ ]:
vote_intention_df.to_csv('data/2024_anes_pre-election_preprocessed.csv')